# 04 — Order Data Analytics: Chipotle Sales & Business Metrics
> **Interview Prep & Technical Mastery Guide**
> 
> *A comprehensive, battle-tested reference for Python Data Science, Business Intelligence, and Analytics Interviews.*

---

## 📌 Executive Summary & Interview Expectations
In technical interviews for Product Analytics, Data Science, and Business Intelligence roles, transaction datasets (like this Chipotle order history) are the **#1 most common scenario**. Interviewers evaluate whether you can:
1. **Ingest Tab-Separated Data**: Parsing `.tsv` files and handling delimiter subtleties.
2. **Execute Targeted GroupBys**: Avoiding the dangerous anti-pattern `df.groupby().sum()['col']` in favor of `df.groupby()['col'].sum()`.
3. **Clean Currency Strings Efficiently**: Replacing slow, brittle `apply(lambda x: float(x[1:-1]))` with vectorized `.str` cleaning.
4. **Detect Real-World Data Traps**: Recognizing whether `item_price` is a **unit price** or a **line-item total** before calculating revenue.
5. **Compute Key E-Commerce Metrics**: Average Order Value (AOV), Basket Size, and Order-Level aggregations.
6. **Select Cardinality Tools Idiomatically**: Using `.nunique()` instead of convoluted `value_counts().count()`.

## 1. Environment Setup & Ingestion
The dataset is tab-delimited (`sep='\t'`). We implement an automated online fallback so this notebook executes reliably in any environment.

In [1]:
import os
import time
import numpy as np
import pandas as pd

# Robust loader: load local chipotle.tsv or fall back to public repository mirror
tsv_path = "chipotle.tsv"
if not os.path.exists(tsv_path):
    tsv_path = "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/chipotle.tsv"

chipo = pd.read_csv(tsv_path, sep="\t")
print(f"Data successfully loaded. Shape: {chipo.shape}")

Data successfully loaded. Shape: (4622, 5)


## 2. Preliminary Data Inspection: `head()`, `info()`, and Schema

In [2]:
# Inspect the first 10 order records
chipo.head(10)

,order_id,quantity,item_name,choice_description,item_price
0,1,1,Chips and Fresh Tomato Salsa,NaN,$2.39
1,1,1,Izze,[Clementine],$3.39
2,1,1,Nantucket Nectar,[Apple],$3.39
3,1,1,Chips and Tomatillo-Green Chili Salsa,NaN,$2.39
4,2,2,Chicken Bowl,"[Tomatillo-Red Chili Salsa (Hot), [Black Beans...",$16.98
5,3,1,Chicken Bowl,"[Fresh Tomato Salsa (Mild), [Rice, Cheese, Sou...",$10.98
6,3,1,Side of Chips,NaN,$1.69
7,4,1,Steak Burrito,"[Tomatillo Red Chili Salsa, [Fajita Vegetables...",$11.75
8,4,1,Steak Soft Tacos,"[Tomatillo Green Chili Salsa, [Pinto Beans, Ch...",$9.25
9,5,1,Steak Burrito,"[Fresh Tomato Salsa, [Rice, Black Beans, Pinto...",$9.25


In [3]:
# Dimensionality and column attributes
print(f"Number of observations (rows): {chipo.shape[0]}")
print(f"Number of columns:             {chipo.shape[1]}")
print(f"Column names:                  {list(chipo.columns)}")
print(f"Index type:                    {type(chipo.index)}")

Number of observations (rows): 4622
Number of columns:             5
Column names:                  ['order_id', 'quantity', 'item_name', 'choice_description', 'item_price']
Index type:                    <class 'pandas.RangeIndex'>


In [4]:
# Schema and null-value check
chipo.info()

<class 'pandas.DataFrame'>
RangeIndex: 4622 entries, 0 to 4621
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   order_id            4622 non-null   int64
 1   quantity            4622 non-null   int64
 2   item_name           4622 non-null   str  
 3   choice_description  3376 non-null   str  
 4   item_price          4622 non-null   str  
dtypes: int64(2), str(3)
memory usage: 180.7 KB


## 3. Targeted GroupBy: Identifying Top-Selling Items

### ⚠️ Top Interview Anti-Pattern: Where to Slice in `groupby`
Compare these two approaches:
```python
# ❌ ANTI-PATTERN: Sums ALL columns first (concatenates strings, sums order_ids!)
chipo.groupby("item_name").sum()["quantity"]

# ✅ IDIOMATIC & EFFICIENT: Slices the target column BEFORE aggregating
chipo.groupby("item_name")["quantity"].sum()
```
**Why this matters in interviews**:
1. Summing the entire DataFrame forces Pandas to aggregate every column, including non-relevant numeric columns (like `order_id`) and strings (concatenating strings across millions of rows can cause severe memory bloat).
2. Slicing upfront (`['quantity']`) isolates only the required series before grouping, saving CPU cycles and memory.

In [5]:
# Top 5 most-ordered items by total quantity
top_items = (
    chipo.groupby("item_name")["quantity"]
    .sum()
    .sort_values(ascending=False)
)

print(f"🥇 Most-ordered item: '{top_items.index[0]}' with {top_items.iloc[0]} units sold")
top_items.head(5)

🥇 Most-ordered item: 'Chicken Bowl' with 761 units sold


item_name
Chicken Bowl           761
Chicken Burrito        591
Chips and Guacamole    506
Steak Burrito          386
Canned Soft Drink      351
Name: quantity, dtype: int64

In [6]:
# Most frequent choice descriptions (custom toppings / salsa)
top_choices = (
    chipo.groupby("choice_description")["quantity"]
    .sum()
    .sort_values(ascending=False)
)

print(f"🥇 Most popular choice: '{top_choices.index[0]}' with {top_choices.iloc[0]} orders")
top_choices.head(5)

🥇 Most popular choice: '[Diet Coke]' with 159 orders


choice_description
[Diet Coke]                                                               159
[Coke]                                                                    143
[Sprite]                                                                   89
[Fresh Tomato Salsa, [Rice, Black Beans, Cheese, Sour Cream, Lettuce]]     49
[Fresh Tomato Salsa, [Rice, Black Beans, Cheese, Sour Cream]]              42
Name: quantity, dtype: int64

In [7]:
# Total items ordered across the entire dataset
total_items = chipo["quantity"].sum()
print(f"Total items ordered: {total_items:,}")

Total items ordered: 4,972


## 4. Currency Cleaning: Vectorized `.str` vs `.apply(lambda)`

### ⚠️ Top Interview Trap: The Brittle `x[1:-1]` Slice & `.apply()` Overhead
In the original exercise, price was cleaned using:
```python
dollarizer = lambda x: float(x[1:-1])
chipo['item_price'] = chipo['item_price'].apply(dollarizer)
```
**Why this is dangerous in production**:
1. **Trailing Whitespace Trap**: `x[1:-1]` assumes there is a leading `$` AND a trailing space (`$2.39 ` $\to$ `'2.39'`). If a clean string without trailing space arrives (`$2.39`), `x[1:-1]` strips the last digit `9` $\to$ `'2.3'` (silently corrupting $2.39 into $2.30)!
2. **Python Bytecode Overhead**: `.apply(lambda)` invokes Python interpreter overhead for every single row ($N$ function calls).
3. **The Vectorized Fix**: `.str.replace('$', '', regex=False).str.strip().astype(float)` executes in compiled C/Rust, handles varying whitespace safely, and is up to **50x faster**!

In [8]:
# Inspect initial item_price dtype
print("Initial item_price dtype:", chipo["item_price"].dtype)
print("Sample raw values:       ", list(chipo["item_price"].head(3)))

Initial item_price dtype: str
Sample raw values:        ['$2.39 ', '$3.39 ', '$3.39 ']


In [9]:
# Safe, idempotent vectorized currency cleaning
if not pd.api.types.is_numeric_dtype(chipo["item_price"]):
    chipo["item_price"] = (
        chipo["item_price"]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.strip()
        .astype(float)
    )

print("Cleaned item_price dtype:", chipo["item_price"].dtype)
chipo[["item_name", "quantity", "item_price"]].head()

Cleaned item_price dtype: float64


,item_name,quantity,item_price
0,Chips and Fresh Tomato Salsa,1,2.39
1,Izze,1,3.39
2,Nantucket Nectar,1,3.39
3,Chips and Tomatillo-Green Chili Salsa,1,2.39
4,Chicken Bowl,2,16.98


## 5. ⚠️ Real-World Interview Trap: Unit Price vs Line-Item Price

In many interviews using the Chipotle dataset, candidates are asked:
> *"How much was total revenue?"*

Many candidates immediately write:
```python
revenue = (chipo['quantity'] * chipo['item_price']).sum()
```
**HOWEVER, inspect the data closely!** Let's check orders where `quantity > 1`:

In [10]:
# Data Validation: Inspect multiple-quantity purchases
multi_items = chipo[chipo["quantity"] > 1][["order_id", "quantity", "item_name", "item_price"]].head(6)
multi_items

,order_id,quantity,item_name,item_price
4,2,2,Chicken Bowl,16.98
18,9,2,Canned Soda,2.18
51,23,2,Canned Soda,2.18
135,60,2,Chicken Salad Bowl,22.50
148,67,2,Steak Burrito,17.98
150,68,2,Chicken Burrito,17.50


### 💡 Key Takeaway:
Look at `order_id = 9`, `quantity = 2`, `item_name = "Canned Soda"`, `item_price = 2.18`.
A single Canned Soda is **$1.09**; two Canned Sodas is **$2.18**!
`item_price` is **ALREADY the total extended price for that line item**!
- If you compute `chipo['quantity'] * chipo['item_price']`, you are **multiplying quantity twice**!
- **True Total Revenue**: `chipo['item_price'].sum()` ($39,237.02)
- **Double-Multiplied Calculation**: `(chipo['quantity'] * chipo['item_price']).sum()` ($34,500.15 - $39,237+ depending on calculation)

*In technical interviews, proactively pointing this out demonstrates exceptional business acumen and data validation diligence.*

In [11]:
# 1. True Revenue (item_price is already line-item total)
true_revenue = chipo["item_price"].sum()

# 2. Author's exercise formula (assuming item_price was unit price)
formula_revenue = (chipo["quantity"] * chipo["item_price"]).sum()

print(f"True Business Revenue:        ${true_revenue:,.2f}")
print(f"Double-Multiplied Revenue:    ${formula_revenue:,.2f}")

True Business Revenue:        $34,500.16
Double-Multiplied Revenue:    $39,237.02


## 6. Order-Level Metrics & Average Order Value (AOV)

### ⚠️ Top Interview Metric: Calculating Total Orders
- Anti-pattern: `chipo.order_id.value_counts().count()` (slow, creates intermediate Series).
- Idiomatic: `chipo['order_id'].nunique()` ($O(N)$ hash set count).

In [12]:
num_orders = chipo["order_id"].nunique()
print(f"Total Unique Orders: {num_orders:,}")

Total Unique Orders: 1,834


### Calculating Average Order Value (AOV):
There are two idiomatic ways to calculate AOV:
1. **GroupBy Order ID**: Sum the revenue per order, then take the mean across orders.
2. **Direct Ratio**: Total Revenue divided by Total Unique Orders.

In [13]:
# Method 1: GroupBy order_id then calculate mean
revenue_per_order = chipo.groupby("order_id")["item_price"].sum()
aov_method1 = revenue_per_order.mean()

# Method 2: Total Revenue / Total Unique Orders
aov_method2 = chipo["item_price"].sum() / chipo["order_id"].nunique()

print(f"Average Order Value (Method 1): ${aov_method1:.2f}")
print(f"Average Order Value (Method 2): ${aov_method2:.2f}")

Average Order Value (Method 1): $18.81
Average Order Value (Method 2): $18.81


In [14]:
# Number of distinct menu items sold
distinct_items = chipo["item_name"].nunique()
print(f"Number of distinct items sold: {distinct_items}")

Number of distinct items sold: 50


## 7. E-Commerce Order Analytics Cheat Sheet

| Business Question | Idiomatic Pandas Expression | Key Consideration |
| :--- | :--- | :--- |
| **Total Orders** | `df['order_id'].nunique()` | Faster and cleaner than `value_counts().count()` |
| **Units per Item** | `df.groupby('item')['qty'].sum()` | Slices `['qty']` before `.sum()` |
| **Safe Currency Cleaning** | `.str.replace('$', '', regex=False).str.strip().astype(float)` | Robust to trailing whitespace; handles vectorization |
| **Total Revenue** | `df['item_price'].sum()` | Verify whether price is per-unit or per-line-item |
| **Average Order Value** | `df.groupby('order_id')['price'].sum().mean()` | Two-stage aggregation (sum then mean) |
| **Average Basket Size** | `df.groupby('order_id')['qty'].sum().mean()` | Units per order |

---
## 🎯 8. Technical Interview Corner: Tricky Questions & Drills

### Q1: The GroupBy Aggregation Performance Trap
**Question**: An interviewer asks: *"What is the computational and memory difference between `df.groupby('A').sum()['B']` and `df.groupby('A')['B'].sum()`?"*

**Answer**:
- `df.groupby('A').sum()['B']`:
  1. Computes the `.sum()` aggregation over **every column in the DataFrame**.
  2. For numeric columns, it computes sums. For string columns, it concatenates strings. For ID columns, it produces meaningless sums.
  3. Constructs a full intermediate DataFrame containing all aggregated columns.
  4. Only then indexes column `'B'`.
- `df.groupby('A')['B'].sum()`:
  1. Restricts the groupby operation strictly to the `SeriesGroupBy` object for column `'B'`.
  2. Computes the sum only for column `'B'`, bypassing all other columns.
  3. Uses significantly less memory and runs drastically faster.

In [15]:
# Demonstration: benchmarking column selection before vs after
t0 = time.perf_counter()
res_after = chipo.groupby("item_name").sum(numeric_only=True)["quantity"]
t1 = time.perf_counter()

t2 = time.perf_counter()
res_before = chipo.groupby("item_name")["quantity"].sum()
t3 = time.perf_counter()

print(f"Time (aggregate all, then slice): {(t1 - t0)*1000:.3f} ms")
print(f"Time (slice column upfront):       {(t3 - t2)*1000:.3f} ms")

Time (aggregate all, then slice): 0.799 ms
Time (slice column upfront):       0.301 ms


### Q2: Why is `.apply()` considered an anti-pattern when vectorized alternatives exist?
**Question**: When cleaning strings or converting currency columns, why should you avoid `df['col'].apply(lambda x: float(x[1:]))`?

**Answer**:
1. **No True Vectorization**: `.apply()` with a Python callable executes an iterative Python `for` loop under the hood. It incurs Python bytecode interpreter overhead on every row.
2. **GIL Bound**: Python callables cannot release the Global Interpreter Lock (GIL).
3. **Vectorized `.str` Methods**: Vectorized string accessors (`.str.replace()`, `.astype()`) operate on contiguous memory buffers implemented in C / PyArrow / Rust, making them **10x to 50x faster** on medium to large datasets.

### Q3: How to Derive True Unit Price
**Question**: Since `item_price` is the total line price for `quantity` units, write a vectorized expression to create a `unit_price` column and identify the single most expensive menu item at Chipotle.

In [16]:
# Calculating true unit price
chipo["unit_price"] = (chipo["item_price"] / chipo["quantity"]).round(2)

# Top 5 most expensive distinct items by unit price
most_expensive = (
    chipo.drop_duplicates(subset=["item_name", "choice_description"])
    .sort_values(by="unit_price", ascending=False)
    [["item_name", "choice_description", "unit_price"]]
    .head(5)
)

most_expensive

,item_name,choice_description,unit_price
1326,Barbacoa Salad Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Rice,...",11.89
2957,Steak Salad Bowl,"[Fresh Tomato Salsa, [Black Beans, Cheese, Gua...",11.89
3120,Steak Salad Bowl,"[Roasted Chili Corn Salsa, [Fajita Vegetables,...",11.89
4313,Steak Salad Bowl,"[Roasted Chili Corn Salsa, [Fajita Vegetables,...",11.89
3208,Barbacoa Salad Bowl,"[Tomatillo Red Chili Salsa, [Black Beans, Chee...",11.89


### Q4: Hands-on Interview Coding Challenge: Order Basket Analysis
**Challenge**: In a single chained expression, calculate:
1. The **percentage of orders** that contained at least one "Chicken Bowl".
2. The **Average Order Value (AOV)** for orders with a "Chicken Bowl" vs orders without one.

In [17]:
# Coding Challenge: Orders with vs without Chicken Bowl
order_summary = (
    chipo.assign(has_chicken_bowl=chipo["item_name"] == "Chicken Bowl")
    .groupby("order_id")
    .agg(
        order_total=("item_price", "sum"),
        contains_chicken_bowl=("has_chicken_bowl", "any")
    )
)

pct_with_bowl = (order_summary["contains_chicken_bowl"].mean()) * 100
aov_comparison = order_summary.groupby("contains_chicken_bowl")["order_total"].mean().round(2)

print(f"Percentage of orders containing a Chicken Bowl: {pct_with_bowl:.1f}%")
print("\nAverage Order Value (AOV) Comparison:")
print(f"  Orders WITH Chicken Bowl:    ${aov_comparison[True]:.2f}")
print(f"  Orders WITHOUT Chicken Bowl: ${aov_comparison[False]:.2f}")

Percentage of orders containing a Chicken Bowl: 33.5%

Average Order Value (AOV) Comparison:
  Orders WITH Chicken Bowl:    $20.84
  Orders WITHOUT Chicken Bowl: $17.79
